In [8]:
!git clone https://github.com/wenerd569/Speech-recognition.git
!curl http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz --output speech_commands_v0.02.tar.gz


Cloning into 'Speech-recognition'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 202 (delta 0), reused 3 (delta 0), pack-reused 190 (from 1)
Receiving objects: 100% (202/202), 41.00 KiB | 583.00 KiB/s, done.
Resolving deltas: 100% (104/104), done.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2316M  100 2316M    0     0  21.6M      0  0:01:47  0:01:47 --:--:-- 21.6M


In [9]:

!mkdir -p ./Speech-recognition/data/speech_commands_v0.02
!tar -xzf speech_commands_v0.02.tar.gz -C Speech-recognition/data/speech_commands_v0.02



In [29]:
%cd /contrent/Speech-recognition
!apt install -y ffmpeg
!pwd
!git fetch
!git pull
!pip install -r requirements.txt



[Errno 2] No such file or directory: '/contrent/Speech-recognition'
/content/Speech-recognition
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
/content/Speech-recognition
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 30 (delta 17), reused 26 (delta 17), pack-reused 0 (from 0)
Unpacking objects: 100% (30/30), 9.52 KiB | 812.00 KiB/s, done.
From https://github.com/wenerd569/Speech-recognition
   9758fbb..27c0b6e  main       -> origin/main
 * [new branch]      olya       -> origin/olya
   b28f56f..a15fe4f  wenerd3    -> origin/wenerd3
Updating 9758fbb..27c0b6e
Fast-forward
 requirements.txt           |   2 +-
 src/heads/attention_net.py |  36 +++++++++
 src/run.ipynb              | 193 ++++++++++++++++++++++++++----------

In [44]:
!mkdir /content/Speech-recognition/data/data_proccessed

!ls /content/Speech-recognition/data/data_proccessed/

mkdir: cannot create directory ‘/content/Speech-recognition/data/data_proccessed’: File exists
DF.csv


In [41]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, DataLoader

from src.train_evaluate import train

from src.data_preparing import MelSpecDataset, init_datasets
from src.base_net import BaseNet
from src.union_net import UnionNet

from src.wav_preproccess import transform_wavs_to_tensors


In [ ]:
DIR_CURRENT = os.path.abspath(os.getcwd())
DIR_TO_UPLOAD_FROM = os.path.join(DIR_CURRENT, "data/speech_commands_v0.02")
DIR_TO_SAVE_TO = os.path.join(DIR_CURRENT, "data/data_proccessed")
DF_PATH = os.path.join(DIR_TO_SAVE_TO, "DF.csv")

print(DIR_CURRENT)

Sample_rate = 16000
N_fft = 1024
N_mels = 128
Hop_length = 128
Target_T = Sample_rate // Hop_length + 1

transform_wavs_to_tensors(DIR_TO_UPLOAD_FROM, DIR_TO_SAVE_TO, Sample_rate, N_fft, N_mels, Hop_length)
init_datasets(DIR_TO_UPLOAD_FROM, DIR_TO_SAVE_TO)

/content/Speech-recognition/src
created melspectrogram model
creating df


FileNotFoundError: [Errno 2] No such file or directory: '/content/Speech-recognition/src/data/speech_commands_v0.02/testing_list.txt'

In [ ]:
BATCH_SIZE = 32
LABELS = sorted(pd.read_csv(DF_PATH)["label"].unique())
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)

BASE_NET_INPUT_SHAPE = (32, 1, N_mels, Target_T)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

def get_loader(df_csv_name: str):
    df_train = pd.read_csv(os.path.join(DIR_TO_SAVE_TO, df_csv_name))
    dataset = MelSpecgDataset(df_train, DIR_TO_SAVE_TO, LABEL_TO_INDEX)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    return train_loader


In [ ]:
optimizer = Adam(net.parameters(), lr=0.001     )
loss = nn.NLLLoss()


In [ ]:



base_cnn = BaseNet(kernel_sizes=[3, 3, 3, 3], strides=[1, 1, 1, 1], paddings=[1, 1, 1, 1])
BASE_NET_OUTPUT_SHAPE = summary(base_cnn.features, input_size = BASE_NET_INPUT_SHAPE, verbose=0).summary_list[-1].output_size
   
mean_net = MeanNet(output_features=len(LABEL_TO_INDEX), input_features=BASE_NET_OUTPUT_SHAPE[1]*BASE_NET_OUTPUT_SHAPE[2])

net = UnionNet(
    base_cnn,
    mean_net,
)

train_loader = get_loader("df_train.csv")
validation_loader = get_loader("df_validation.csv")

train(net, epochs=40, train_loader=train_loader, validate_loader=validation_loader, optimizer=optimizer, loss=loss, num_classes=NUM_CLASSES, device=device)


NameError: name 'DF_PATH' is not defined